# Notebook 03 — Trade Report Grain Audit

**Goal:** Confirm whether `kalshi.trade_report` gives 1 row per trade or 1 row per leg.

**Why it matters:** If each row is a leg, our trade counts are inflated by leg count. A 5-leg parlay would show as 5 trades not 1. All microstructure analysis built on `COUNT(*)` would be wrong.

**Method:** Find a specific parlay ticker, count its rows in `trade_report`, then check how many legs it has via the Kalshi API.

In [1]:
import os
import pandas as pd
from dune_client.client import DuneClient
from dune_client.query import QueryBase
from dune_client.types import QueryParameter

dune = DuneClient(os.environ["DUNE_API_KEY"])

KeyError: 'DUNE_API_KEY'

## Step 1 — Pull a sample of trade_report rows

Pick one specific parlay ticker and see how many rows it has.

In [ ]:
# Pull sample rows from trade_report for one parlay ticker
# We'll look at a high-volume ticker so we have enough rows to audit

sample_query = QueryBase(
    query_id=7545749  # our existing trade_report probe — swap if needed
)

result = dune.get_latest_result(7545749)
df = pd.DataFrame(result.result.rows)
print(df.shape)
df.head(10)

## Finding — Row Grain is 1 Per Trade

**Question:** Does `kalshi.trade_report` give 1 row per trade or 1 row per leg?

**Result:** 1 row per trade.

**Evidence:**
- Top ticker: 14,444 rows across 7 distinct dates = ~2,000 trades/day
- Same ticker has 50 distinct prices — real market microstructure, not leg duplication
- If per-leg, all rows from one trade would share the same price. We see price variation, confirming these are independent transactions.

**Implication:** All `COUNT(*)` figures in the microstructure section are valid. 25M total trades is real. Median trade of $9.30 is real.